# Module 4 Integration Challenge
## Claims & Provider Network Assistant

**Duration:** 60 minutes | **Format:** solo or pairs

See the accompanying brief (`Module4_Integration_Challenge_Brief.md`) for the full requirements
checklist and rubric. This notebook gives you the environment setup, mock data, and test questions —
everything else is yours to build.

**Reminder of the non-agentic scope:** your function dispatch should be a single detection + single
execution + single follow-up call. No loops, no chained multi-function calls, no re-planning.


## Setup: Azure OpenAI Environment

**Facilitator talking points (5 min):**
- We use **Azure OpenAI** endpoints for both the chat model (GPT) and the embedding model, configured entirely via a `.env` file — never hardcode keys/endpoints in code.
- Two separate *deployments* are typical in Azure: one for chat, one for embeddings — hence two deployment-name variables.
- This mirrors real enterprise practice: config is environment-driven so the same code runs across dev/test/prod by swapping `.env` files.

Create a `.env` file (not committed to source control) alongside this notebook with:

```
AZURE_OPENAI_API_KEY=your-key-here
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_VERSION=2024-10-21
AZURE_OPENAI_CHAT_DEPLOYMENT=your-gpt-deployment-name
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=your-embedding-deployment-name
```


In [ ]:
# =============================================================================
# SHARED SETUP — run this cell first (not an activity, just plumbing)
# =============================================================================
import os
import json
import time
import uuid
import random
import logging
from datetime import datetime, timezone
from typing import Optional, Any

from dotenv import load_dotenv
from openai import AzureOpenAI, APITimeoutError, APIConnectionError, RateLimitError, APIError

# --- Load environment variables from .env ---
load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

assert AZURE_OPENAI_API_KEY, "Missing AZURE_OPENAI_API_KEY in .env"
assert AZURE_OPENAI_ENDPOINT, "Missing AZURE_OPENAI_ENDPOINT in .env"
assert CHAT_DEPLOYMENT, "Missing AZURE_OPENAI_CHAT_DEPLOYMENT in .env"

# --- Azure OpenAI client (one client handles both chat + embeddings) ---
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("Azure OpenAI client initialised.")
print(f"Chat deployment:      {CHAT_DEPLOYMENT}")
print(f"Embedding deployment: {EMBEDDING_DEPLOYMENT}")


In [ ]:
# --- Quick sanity check: confirm connectivity before we build anything on top of it ---
sanity_response = client.chat.completions.create(
    model=CHAT_DEPLOYMENT,
    messages=[{"role": "user", "content": "Reply with exactly: connection OK"}],
    max_tokens=10,
    temperature=0,
)
print(sanity_response.choices[0].message.content)


---
## Mock data (given — do not need to modify)

Two data sources stand in for real backend systems: a claims database and a provider network directory.


In [ ]:
# --- Mock claims data ---
CLAIMS_DB = {
    "CLM-1001": {"status": "approved", "amount_billed": 250.00, "amount_approved": 200.00, "member_id": "MEM-01"},
    "CLM-1002": {"status": "pending_review", "amount_billed": 900.00, "amount_approved": None, "member_id": "MEM-02"},
    "CLM-1003": {"status": "denied", "amount_billed": 150.00, "amount_approved": 0.00, "member_id": "MEM-01"},
}

# --- Mock provider network directory ---
PROVIDER_DB = {
    "NPI-2001": {"name": "Dr. A. Rao", "specialty": "Cardiology", "in_network": True},
    "NPI-2002": {"name": "Dr. B. Shah", "specialty": "Dermatology", "in_network": False},
    "NPI-2003": {"name": "Dr. C. Iyer", "specialty": "Orthopedics", "in_network": True},
}


---
## Test question bank (given — run these against your finished `ClaimsAssistant`)

This is the exact set your solution will be checked against at the end. Some are designed to
exercise specific requirements — see the comment on each.


In [ ]:
test_questions = [
    "What is the status of claim CLM-1001?",          # should trigger claim-status function
    "Is provider NPI-2002 in network?",                 # should trigger provider-network function
    "What is a deductible, in general terms?",           # should NOT trigger any function (abstain check)
    "",                                                    # empty input -> must not crash
    "What is the status of claim CLM-9999?",             # unknown claim -> must not crash, honest fallback
    "Is provider NPI-0000 in network?",                  # unknown provider -> must not crash, honest fallback
]


---
## 1. Structured output schema + system prompt

Design your response schema and the system prompt that enforces it. Think about what fields a
claims/provider answer needs (a reference ID, an answer, a numeric value, a status flag, a disclaimer
are a reasonable starting point — but it's your call).


In [ ]:
# TODO: define your response schema (required fields + expected types)
CLAIMS_RESPONSE_SCHEMA = {
    "required_fields": {
        # e.g. "reference_id": str,
    }
}

# TODO: write a system prompt that forces JSON-only output matching your schema
CLAIMS_SYSTEM_PROMPT = """
"""


---
## 2. Hardened API call wrapper

Build `call_llm()` — explicit timeout, retry with exponential backoff + jitter on transient errors only,
raises after max retries.


In [ ]:
def call_llm(prompt: str, system_prompt: str = None, max_retries: int = 3, timeout_seconds: float = 15.0) -> str:
    # TODO: implement the hardened call (timeout + retry-with-backoff on transient errors)
    pass


---
## 3. Validation

Build `validate_response()`: presence check, type check, and at least one business sanity check.


In [ ]:
def validate_response(response: dict, schema: dict = CLAIMS_RESPONSE_SCHEMA) -> tuple:
    # TODO: return (is_valid: bool, errors: list[str])
    pass


---
## 4. Error handling with fallback

Build a function that gets a validated response, but never raises — empty input, unknown claim/provider
ID, and API failures should all resolve to a sensible fallback dict.


In [ ]:
def get_safe_response(question: str) -> dict:
    # TODO: guard against empty/ambiguous input
    # TODO: call the model, validate/repair, and catch failures at each stage
    # TODO: return a fallback dict on any failure - never raise
    pass


---
## 6. Logging & traceability

Build `log_interaction()` writing structured JSON lines with a timestamp, and use a correlation ID
per question to tie related log entries together.


In [ ]:
LOG_FILE = "claims_log.jsonl"

def log_interaction(record: dict) -> None:
    # TODO: add a timestamp if missing, append record as one JSON line to LOG_FILE
    pass


---
## 7. Assemble: `ClaimsAssistant`

Wire everything above into one class with a single public `.ask(question)` method. It should never
raise to the caller, and should log every interaction with a correlation ID.


In [ ]:
class ClaimsAssistant:
    def __init__(self):
        pass

    def ask(self, question: str) -> dict:
        # TODO: guard clause -> function-dispatch attempt -> fallback on failure -> log -> return
        pass


---
## 8. Run the test bank

This is your final check — run every question in `test_questions` and confirm nothing crashes,
the two function-triggering questions actually trigger, the general-knowledge question does NOT
trigger a function, and the two "unknown ID" questions return honest fallbacks.


In [ ]:
assistant = ClaimsAssistant()

for q in test_questions:
    print(f"Q: {q!r}")
    print("A:", assistant.ask(q))
    print()


---
## Self-review

Go back to the rubric checklist in `Module4_Integration_Challenge_Brief.md` and tick off each item
honestly. If anything is unticked, that's exactly what to ask the facilitator about.
